# Notebook 01: Parameter Space Exploration

**Gap 1**: Abarr & Krawczynski (2019) fixed a single configuration.
Here we sweep over (spin, r_bp, beta, phi, inclination) using Latin Hypercube Sampling
and visualize the resulting polarization signature space.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '..')

from src.parameter_sweep.generate_grid import latin_hypercube_sample, run_sweep
from src.utils.physics import PARAM_BOUNDS, ENERGY_BINS
from src.utils.io import load_dataset_hdf5


In [ ]:
# Generate 500-sample LHS grid (small for demo)
params_array, param_names = latin_hypercube_sample(500, PARAM_BOUNDS, seed=42)
print('Parameter names:', param_names)
print('Array shape:', params_array.shape)


In [ ]:
# Run sweep (uses mock simulator — replace with real ray-tracer)
X, y = run_sweep(params_array, param_names, '../data/simulated/demo_sweep.h5')
print('Spectra shape:', X.shape)  # (N_samples, N_energy, 3)
print('Params shape:', y.shape)   # (N_samples, 5)


In [ ]:
# Visualize: polarization angle swing vs misalignment beta
from src.utils.physics import polarization_angle_swing

beta_col = param_names.index('beta')
swings = [polarization_angle_swing(X[i, :, 2], ENERGY_BINS) for i in range(len(X))]
betas = y[:, beta_col]

fig, ax = plt.subplots(figsize=(8, 5))
sc = ax.scatter(betas, swings, c=y[:, param_names.index('spin')], cmap='plasma', alpha=0.5, s=15)
plt.colorbar(sc, label='BH spin')
ax.set_xlabel('Misalignment angle β [°]')
ax.set_ylabel('Polarization angle swing [°]')
ax.set_title('Pol. angle swing vs misalignment across parameter space')
plt.tight_layout()
plt.savefig('../results/figures/parameter_sweep_overview.png', dpi=150)
plt.show()
